# Introducción a xarray y archivos NetCDF

Este notebook muestra cómo abrir, explorar, procesar y graficar datos oceanográficos
en formato **NetCDF** usando la librería **xarray**.

Es el punto de partida recomendado para estudiantes que trabajan por primera vez
con datos de reanálisis (ERA5, CMEMS) o datos satelitales en el laboratorio.

**Requisitos:**
```
pip install xarray numpy matplotlib netCDF4
```

---
*Laboratorio de Oceanografía Física (LOFA) — Universidad Austral de Chile*

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd

print('xarray versión:', xr.__version__)

## 1. Crear un dataset sintético de SST

Generamos datos sintéticos de Temperatura Superficial del Mar (SST) con:
- Un ciclo estacional (más fría en invierno austral)
- Un gradiente espacial (más fría cerca de la costa)
- Ruido aleatorio

Esto simula lo que verías al abrir un archivo ERA5 o MODIS real.

In [ ]:
# Dimensiones del dataset
fechas = pd.date_range('2020-01-01', '2022-12-31', freq='MS')  # mensual
lats   = np.arange(-41.0, -39.0, 0.25)   # latitudes de Valdivia
lons   = np.arange(-74.0, -72.0, 0.25)   # longitudes de Valdivia

# SST sintética: temperatura base + ciclo estacional + gradiente + ruido
# El ciclo usa el mes del año para crear estacionalidad austral
meses = np.array([f.month for f in fechas])
ciclo = 2.5 * np.cos(2 * np.pi * (meses - 1) / 12)  # máx en enero (verano austral)

T, La, Lo = np.meshgrid(ciclo, lats, lons, indexing='ij')
gradiente_lat = (La - lats.mean()) * 0.3   # más fría hacia el sur
gradiente_lon = (Lo - lons.mean()) * (-1.5)  # más fría cerca de la costa (oeste)
ruido = np.random.normal(0, 0.3, T.shape)

sst_data = 13.5 + T + gradiente_lat + gradiente_lon + ruido

# Crear el xr.Dataset
ds = xr.Dataset(
    data_vars={
        'sst': xr.DataArray(
            data=sst_data,
            dims=['time', 'lat', 'lon'],
            attrs={'long_name': 'Temperatura Superficial del Mar', 'units': '°C'}
        )
    },
    coords={
        'time': fechas,
        'lat':  xr.DataArray(lats, dims='lat', attrs={'units': 'degrees_north'}),
        'lon':  xr.DataArray(lons, dims='lon', attrs={'units': 'degrees_east'}),
    },
    attrs={
        'title'    : 'SST sintética - región Valdivia',
        'source'   : 'Datos generados para tutorial LOFA',
        'history'  : 'Creado con intro_xarray_netcdf.ipynb',
    }
)

print(ds)

## 2. Explorar la estructura del dataset

Un `xr.Dataset` tiene tres componentes clave:
- **`data_vars`**: las variables con datos (equivale a las capas del NetCDF)
- **`coords`**: las coordenadas (tiempo, latitud, longitud, profundidad…)
- **`attrs`**: metadatos que describen el archivo (título, fuente, unidades)

In [ ]:
# Ver las variables disponibles
print('Variables:', list(ds.data_vars))
print('Coordenadas:', list(ds.coords))
print()

# Ver una variable específica (xr.DataArray)
print(ds['sst'])

In [ ]:
# Acceder a los metadatos (atributos)
print('Metadatos del dataset:')
for k, v in ds.attrs.items():
    print(f'  {k}: {v}')

print()
print('Metadatos de la variable SST:')
print('  long_name:', ds['sst'].attrs['long_name'])
print('  units:', ds['sst'].attrs['units'])

print()
print('Rango temporal:', str(ds.time.values[0])[:10], '→', str(ds.time.values[-1])[:10])
print('Rango latitudes:', float(ds.lat.min()), '→', float(ds.lat.max()))
print('Rango longitudes:', float(ds.lon.min()), '→', float(ds.lon.max()))

## 3. Seleccionar datos

xarray ofrece dos formas de seleccionar datos:
- **`.sel()`** — selecciona por valor de coordenada (lo más común)
- **`.isel()`** — selecciona por posición de índice (como NumPy)

In [ ]:
# Seleccionar un punto específico (lat, lon más cercana)
punto = ds['sst'].sel(lat=-40.0, lon=-73.5, method='nearest')
print('SST en lat=-40°, lon=-73.5° (serie temporal):')
print(punto.values[:6], '...')  # primeros 6 valores

print()

# Seleccionar un mes específico
enero_2021 = ds['sst'].sel(time='2021-01')
print('SST en enero 2021 (mapa):', enero_2021.shape)

In [ ]:
# Seleccionar un rango (slice) de fechas y región
subregion = ds['sst'].sel(
    time=slice('2021-01-01', '2021-06-30'),
    lat=slice(-41.0, -40.0),
    lon=slice(-74.0, -73.0)
)

print('Subregión seleccionada:', subregion.shape)
print('Dimensiones:', dict(subregion.dims))

## 4. Estadísticas básicas

xarray permite calcular estadísticas colapsando una o varias dimensiones.

In [ ]:
# Media temporal (colapsa el tiempo → mapa)
sst_media = ds['sst'].mean(dim='time')
print('Media temporal — shape:', sst_media.shape)

# Media espacial (colapsa lat y lon → serie temporal)
sst_serie = ds['sst'].mean(dim=['lat', 'lon'])
print('Media espacial — shape:', sst_serie.shape)

# Climatología mensual (ciclo anual promedio)
climatologia = ds['sst'].groupby('time.month').mean('time')
print('Climatología mensual — shape:', climatologia.shape)

In [ ]:
# Anomalías = serie original - climatología mensual
anomalias = ds['sst'].groupby('time.month') - climatologia
print('Anomalías — shape:', anomalias.shape)
print('Media de anomalías (debería ser ≈ 0):', float(anomalias.mean().round(3)))

## 5. Visualización

xarray tiene integración directa con matplotlib: `.plot()` detecta automáticamente
si es una serie temporal (1D) o un mapa (2D) y elige el tipo de gráfico adecuado.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Serie temporal (media espacial)
sst_serie.plot(ax=axes[0], color='#1F77B4', linewidth=2)
axes[0].set_title('Serie temporal de SST\n(promedio espacial)')
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('SST (°C)')

# Gráfico 2: Mapa (promedio temporal)
sst_media.plot(ax=axes[1], cmap='RdYlBu_r',
               cbar_kwargs={'label': 'SST (°C)'})
axes[1].set_title('SST media temporal\n(promedio 2020–2022)')

plt.tight_layout()
plt.show()

In [ ]:
# Climatología mensual
sst_clim_espacial = climatologia.mean(dim=['lat', 'lon'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, 13), sst_clim_espacial.values, color='#1F77B4', alpha=0.7)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
ax.set_ylabel('SST media (°C)')
ax.set_title('Climatología mensual de SST', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Guardar en NetCDF

Para guardar un dataset modificado como nuevo archivo `.nc`:

In [ ]:
# Agregar las anomalías como nueva variable y guardar
ds['sst_anomalia'] = anomalias
ds['sst_anomalia'].attrs = {'long_name': 'Anomalía de SST', 'units': '°C'}

# ds.to_netcdf('sst_con_anomalias.nc')  # Descomenta para guardar
print('Dataset listo para guardar:', list(ds.data_vars))
print('Para guardar: ds.to_netcdf("archivo_salida.nc")')

---
## Resumen de comandos clave

| Tarea | Comando |
|---|---|
| Abrir archivo | `xr.open_dataset('archivo.nc')` |
| Ver estructura | `print(ds)` o `ds` en Jupyter |
| Seleccionar por valor | `ds['var'].sel(lat=-40.0, method='nearest')` |
| Seleccionar rango | `ds['var'].sel(time=slice('2020', '2021'))` |
| Promedio temporal | `ds['var'].mean(dim='time')` |
| Climatología mensual | `ds['var'].groupby('time.month').mean('time')` |
| Anomalías | `ds['var'].groupby('time.month') - climatologia` |
| Graficar | `ds['var'].plot()` |
| Guardar | `ds.to_netcdf('salida.nc')` |